In [5]:
import time
import torch
import numpy as np
import os
from luxai_s3.utils import to_numpy

# Import your custom env, wrappers, and agent
from environment.lux_s3_gymnasium_env import LuxCustomGymEnv
from environment.observation_wrapper import FlattenNormalizeObservation
from ppo_agent.agent import PPOAgent

In [8]:
config = {
    'seed': 42,
    'total_timesteps': 1001500,
    'num_steps_per_rollout': 256, # Steps collected before update
    'learning_rate': 3e-4,
    'gamma': 0.99,
    'lambda_': 0.95, # GAE lambda
    'num_epochs': 10, # PPO epochs per update
    'batch_size': 64, # PPO minibatch size
    'clip_coef': 0.2, # PPO clip coefficient
    'vf_coef': 0.5,   # Value function loss coefficient
    'ent_coef': 0.01, # Entropy bonus coefficient
    'sap_coef': 0.5,  # Sap target loss coefficient
    'use_gpu': True,
    'hidden_dim': 256,
    'log_interval': 10,
    'save_interval': 100,
    'save_path': 'models/lux_ppo_single_env_agent' # Updated save path name
}

In [9]:
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
if config['use_gpu'] and torch.cuda.is_available():
    torch.cuda.manual_seed_all(config['seed'])
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    config['use_gpu'] = False # Ensure config reflects actual device used

# --- Single Environment Setup ---
base_env = LuxCustomGymEnv(random_seed=config['seed'])
env = FlattenNormalizeObservation(base_env)

# Reset the env to get initial state and spaces
initial_obs_dict, initial_info = env.reset(seed=config['seed'])
obs_space = env.observation_space
action_space = env.unwrapped.action_space

print(f"Observation Space (Flattened/Normalized): {obs_space}") #
print(f"Action Space (Original Dict): {action_space}") #

# Agent Setup
agent_config = config.copy()
agent = PPOAgent(
    num_envs=1, # Set num_envs to 1
    obs_space_shape=obs_space.shape, # Use the flattened shape from the wrapped env
    action_space=action_space,       # Use the original single-player Dict action space
    config=agent_config
) #

agent.load_checkpoint('checkpoints/checkpoint_step_900000.pt')
agent.network.to(device) # Ensure network is on the correct device #
# The buffer inside the agent is initialized with num_envs=1
print(f"Agent network: {agent.network}") #

Observation Space (Flattened/Normalized): Box(-1.0, 1.0, (6064,), float32)
Action Space (Original Dict): Dict('action_type': MultiDiscrete([6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6]), 'sap_target': Box(-7, 7, (16, 2), int16))
Checkpoint loaded from checkpoints/checkpoint_step_900000.pt
Agent network: ActorCritic(
  (shared_layers): Sequential(
    (0): Linear(in_features=6064, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (action_type_head): Linear(in_features=256, out_features=96, bias=True)
  (sap_target_head): Linear(in_features=256, out_features=32, bias=True)
  (value_head): Linear(in_features=256, out_features=1, bias=True)
)


In [10]:
# --- Training Loop ---
total_points_player_0_log = 0
episode_points_p0 = 0
total_episodes = 0

# Adjust total updates calculation based on single env
num_updates = config['total_timesteps'] // config['num_steps_per_rollout']
global_step = 0

CHECKPOINT_INTERVAL = 100000
save_dir = "checkpoints" # Define save directory
os.makedirs(save_dir, exist_ok=True) # Create directory if it doesn't exist

print(f"Starting training for {config['total_timesteps']} timesteps...")
print(f"Number of updates: {num_updates}")
print(f"Saving checkpoints every {CHECKPOINT_INTERVAL} steps to '{save_dir}/'")

# Use the initial observation from the reset call above
obs = initial_obs_dict # Already a numpy array from the wrapper

start_time = time.time()
for update in range(1, num_updates + 1):
    agent.network.train() # Set network to training mode #

    for step in range(config['num_steps_per_rollout']):
        global_step += 1 # Increment by 1 for single env

        # Agent expects a batch dimension, even if it's 1. Unsqueeze the observation.
        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)

        # Get action and value from agent
        actions_dict_batch, action_types_tensor, sap_targets_tensor, log_probs_tensor, values_tensor = agent.get_action_and_value(obs_tensor) #

        # Extract the single action from the batch dimension for the env.step call
        # The env expects {'player_0': single_action_dict, 'player_1': single_action_dict}
        single_actions_dict = {
            'action_type': actions_dict_batch['action_type'][0], # Get first (only) item
            'sap_target': actions_dict_batch['sap_target'][0]   # Get first (only) item
        }

        action_for_env = {
            'player_0': single_actions_dict,
            'player_1': single_actions_dict # Assuming the same action for both players
        }

        # Step the env
        next_obs, rewards_dict, terminated_dict, truncated_dict, infos = env.step(action_for_env) #

        # CHeckpoint saving
        if global_step % CHECKPOINT_INTERVAL == 0:
            checkpoint_filepath = os.path.join(save_dir, f"run_2_checkpoint_step_{global_step}.pt")
            agent.save_checkpoint(checkpoint_filepath)

        if global_step % 1000 == 0: # Log every 200 global steps
            p0_reward = rewards_dict.get('player_0', 0.0)
            p1_reward = rewards_dict.get('player_1', 0.0)
            print(f"Step {global_step}: Shaped Rewards -> P0={p0_reward:.4f}, P1={p1_reward:.4f}")

        reward = rewards_dict.get('player_0', 0.0) # Default to 0 if key missing

        # Combine terminated/truncated flags (True if either is True for any player)
        terminated = any(terminated_dict.values())
        truncated = any(truncated_dict.values())
        done = terminated or truncated

        # Add experience to buffer
        agent.buffer.add(
            torch.tensor(obs, dtype=torch.float32).to(device),   # Current obs (single)
            action_types_tensor.squeeze(0),
            sap_targets_tensor.squeeze(0),
            torch.tensor(reward, dtype=torch.float32).to(device),
            torch.tensor([done], dtype=torch.float32).to(device),
            values_tensor.squeeze(0),
            log_probs_tensor.squeeze(0)
        ) #

        obs = next_obs # Update observation

        # Handle episode end
        if done:
            episode_points_p0 = 0
            obs, info = env.reset()


    # --- GAE and PPO Update ---
    agent.network.eval() # Set network to evaluation mode for value calculation
    with torch.no_grad():
        last_obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
        last_value = agent.network.get_value(last_obs_tensor).flatten()

    # Compute advantages and returns using the buffer
    agent.buffer.compute_returns_and_advantages(last_value, config['gamma'], config['lambda_'])
    agent.update()

    # Logging
    if update % config.get('log_interval', 10) == 0:
        print(f"Update {update}/{num_updates}, Global Step {global_step}, Time: {time.time() - start_time:.2f}s")
        # TODO: Add detailed logging if agent.update() returns metrics

env.close() #
print("Training finished.")
print(f"\n--- Training Summary ---")
print(f"Total Episodes: {total_episodes}")
print(f"Total Points Logged (Player 0): {total_points_player_0_log}") # Use the correct total
if total_episodes > 0:
    print(f"Average Points per Episode (Player 0): {total_points_player_0_log / total_episodes:.2f}")

Starting training for 1001500 timesteps...
Number of updates: 3912
Saving checkpoints every 100000 steps to 'checkpoints/'
Step 1000: Shaped Rewards -> P0=-0.2000, P1=-0.4550
Step 2000: Shaped Rewards -> P0=4.8502, P1=4.5733
Update 10/3912, Global Step 2560, Time: 18.51s
Step 3000: Shaped Rewards -> P0=-0.2000, P1=-0.4600
Step 4000: Shaped Rewards -> P0=-0.1750, P1=-0.4550
Step 5000: Shaped Rewards -> P0=-0.1650, P1=-0.4550
Update 20/3912, Global Step 5120, Time: 35.98s
Step 6000: Shaped Rewards -> P0=-0.1100, P1=-0.3350
Step 7000: Shaped Rewards -> P0=-0.1450, P1=-0.2750
Update 30/3912, Global Step 7680, Time: 54.58s
Step 8000: Shaped Rewards -> P0=4.9996, P1=-0.0750
Step 9000: Shaped Rewards -> P0=4.9950, P1=-0.0300
Step 10000: Shaped Rewards -> P0=0.0000, P1=0.0000
Update 40/3912, Global Step 10240, Time: 73.33s
Step 11000: Shaped Rewards -> P0=4.8871, P1=4.7000
Step 12000: Shaped Rewards -> P0=-0.2000, P1=-0.4600
Update 50/3912, Global Step 12800, Time: 92.81s
Step 13000: Shaped Re